In [220]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import calliope

# We increase logging verbosity
calliope.set_log_verbosity("INFO", include_solver_output=True)

model = calliope.read_yaml("model.yaml")

[2025-11-03 00:04:22] INFO     Math init | loading pre-defined math.
[2025-11-03 00:04:22] INFO     Math init | loading math files {'milp', 'spores', 'storage_inter_cluster', 'base', 'operate'}.


[2025-11-03 00:04:22] INFO     Model: preprocessing data
[2025-11-03 00:04:22] INFO     Math build | building applied math with ['base'].
[2025-11-03 00:04:22] INFO     input data `color` not defined in model math; it will not be available in the optimisation problem.
[2025-11-03 00:04:22] INFO     input data `name` not defined in model math; it will not be available in the optimisation problem.
[2025-11-03 00:04:22] INFO     input data `link_to` not defined in model math; it will not be available in the optimisation problem.
[2025-11-03 00:04:22] INFO     input data `link_from` not defined in model math; it will not be available in the optimisation problem.
[2025-11-03 00:04:22] WARNING  ModelWarning: Only one timestep defined. Inferring timestep resolution to be 1 hour

[2025-11-03 00:04:22] INFO     input data `color` not defined in model math; it will not be available in the optimisation problem.
[2025-11-03 00:04:22] INFO     input data `name` not defined in model math; it will no

In [221]:
model.inputs

<xarray.Dataset> Size: 2kB
Dimensions:                     (costs: 1, techs: 8, nodes: 5, carriers: 2,
                                 timesteps: 1)
Coordinates:
  * costs                       (costs) object 8B 'monetary'
  * techs                       (techs) object 64B 'SE1_to_SH1' ... 'supply_g...
  * carriers                    (carriers) object 16B 'electricity' 'heat'
  * nodes                       (nodes) object 40B 'D1' 'D2' 'SE1' 'SH1' 'TH1'
  * timesteps                   (timesteps) datetime64[ns] 8B 2050-01-01
Data variables: (12/22)
    bigM                        float64 8B 1e+06
    objective_cost_weights      (costs) float64 8B 1.0
    base_tech                   (techs) object 64B 'transmission' ... 'supply'
    carrier_in                  (nodes, techs, carriers) bool 80B False ... F...
    color                       (techs) object 64B '#6783E3' ... '#C98AAD'
    name                        (techs) object 64B 'Local electricity transmi...
    ...                          ...
    link_from                   (techs) object 64B 'SH1' 'TH1' 'D1' ... nan nan
    cost_flow_cap_per_distance  (costs, techs) float64 64B 1.0 1.0 ... nan nan
    definition_matrix           (nodes, techs, carriers) bool 80B False ... F...
    distance                    (techs) float64 64B 0.8098 0.6853 ... nan nan
    timestep_resolution         (timesteps) float64 8B 1.0
    timestep_weights            (timesteps) float64 8B 1.0

In [222]:
model.inputs.flow_cap_max.to_series().dropna()

techs
SE1_to_SH1            2000.0
SH1_to_TH1            2000.0
TH1_to_D1             2000.0
TH1_to_D2             2000.0
supply_electricity    2000.0
supply_geothermal     2000.0
Name: flow_cap_max, dtype: float64

In [223]:
model.inputs.sink_use_equals.sum(
    "timesteps", min_count=1, skipna=True
).to_series().dropna()

nodes  techs             
D1     demand_heat           500.0
D2     demand_heat           500.0
SH1    demand_electricity    300.0
TH1    demand_heat             0.0
Name: sink_use_equals, dtype: float64

In [224]:
model.build()
model.solve()

[2025-11-03 00:04:27] INFO     Model: backend build starting
[2025-11-03 00:04:27] INFO     Optimisation Model | parameters/lookups | Generated.
[2025-11-03 00:04:27] INFO     Optimisation Model | variables | Generated.
[2025-11-03 00:04:28] INFO     Optimisation Model | global_expressions | Generated.
[2025-11-03 00:04:29] INFO     Optimisation Model | constraints | Generated.
[2025-11-03 00:04:29] INFO     Optimisation Model | piecewise_constraints | Generated.
[2025-11-03 00:04:29] INFO     Optimisation Model | objectives | Generated.
[2025-11-03 00:04:29] INFO     Model: backend build complete
[2025-11-03 00:04:29] INFO     Optimisation model | starting model in base mode.
[2025-11-03 00:04:30] DEBUG    Set parameter Username
Set parameter LicenseID to value 2716243
[2025-11-03 00:04:30] DEBUG    Academic license - for non-commercial use only - expires 2026-09-30
Read LP format model from file C:\Users\alexn\AppData\Local\Temp\tmpxj2nb6t2.pyomo.lp
Reading time = 0.00 seconds
x1: 50

In [225]:
model.results

<xarray.Dataset> Size: 7kB
Dimensions:                     (nodes: 5, techs: 8, carriers: 2, timesteps: 1,
                                 costs: 1)
Coordinates:
  * techs                       (techs) object 64B 'SE1_to_SH1' ... 'supply_g...
  * nodes                       (nodes) object 40B 'D1' 'D2' 'SE1' 'SH1' 'TH1'
  * carriers                    (carriers) object 16B 'electricity' 'heat'
  * timesteps                   (timesteps) datetime64[ns] 8B 2050-01-01
  * costs                       (costs) object 8B 'monetary'
Data variables: (12/21)
    flow_cap                    (nodes, techs, carriers) float64 640B nan ......
    link_flow_cap               (techs) float64 64B 2e+03 2e+03 ... nan nan
    flow_out                    (nodes, techs, carriers, timesteps) float64 640B ...
    flow_in                     (nodes, techs, carriers, timesteps) float64 640B ...
    source_use                  (nodes, techs, timesteps) float64 320B nan .....
    source_cap                  (nodes, techs) float64 320B nan nan ... nan nan
    ...                          ...
    min_cost_optimisation       float64 8B 1.348e+03
    capacity_factor             (nodes, techs, carriers, timesteps) float64 640B ...
    systemwide_capacity_factor  (techs, carriers) float64 128B 0.075 ... 0.5112
    systemwide_levelised_cost   (techs, costs, carriers) float64 128B 0.01 .....
    total_levelised_cost        (costs, carriers) float64 16B 4.457 1.318
    unmet_sum                   (nodes, carriers, timesteps) float64 80B nan ...

In [226]:
df_heat = (
    model.results.flow_out.sel(carriers="heat")
    .sum("nodes", min_count=1, skipna=True)
    .to_series()
    .dropna()
    .unstack("techs")
)

df_heat.head()

techs,SH1_to_TH1,TH1_to_D1,TH1_to_D2,supply_geothermal
timesteps,,,,
2050-01-01,1008.351014,500.0,500.0,1022.409629


In [227]:
df_electricity = (
    model.results.flow_out.sel(carriers="electricity")
    .sum("nodes", min_count=1, skipna=True)
    .to_series()
    .dropna()
    .unstack("techs")
)

df_electricity.head()

techs,SE1_to_SH1,supply_electricity
timesteps,,
2050-01-01,300.0,302.451666


In [228]:
costs = model.results.cost.to_series().dropna()
costs.head()

nodes  techs               costs   
D1     TH1_to_D1           monetary      5.000000
D2     TH1_to_D2           monetary      5.000000
SE1    SE1_to_SH1          monetary      0.000000
       supply_electricity  monetary    302.451666
SH1    SE1_to_SH1          monetary      3.000000
Name: cost, dtype: float64

In [179]:
# We set the color mapping to use in all our plots by extracting the colors defined in the technology definitions of our model.
colors = model.inputs.color.to_series().to_dict()

df_electricity = (
    (model.results.flow_out.fillna(0) - model.results.flow_in.fillna(0))
    .sel(carriers="electricity")
    .sum("nodes")
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow in/out (kWh)")
    .reset_index()
)
df_electricity_demand = df_electricity[df_electricity.techs == "demand_electricity"]
df_electricity_other = df_electricity[df_electricity.techs != "demand_electricity"]

print(df_electricity.head())

fig1 = px.bar(
    df_electricity_other,
    x="timesteps",
    y="Flow in/out (kWh)",
    color="techs",
    color_discrete_map=colors,
)
fig1.add_scatter(
    x=df_electricity_demand.timesteps,
    y=-1 * df_electricity_demand["Flow in/out (kWh)"],
    marker_color="black",
    name="demand",
)

                techs  timesteps  Flow in/out (kWh)
0          SE1_to_SH1 2050-01-01          -2.451666
1  demand_electricity 2050-01-01        -300.000000
2  supply_electricity 2050-01-01         302.451666


In [180]:
carriers = ["heat", "electricity"]
df_flows = (
    (model.results.flow_out.fillna(0) - model.results.flow_in.fillna(0))
    .sel(carriers=carriers)
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow in/out (kWh)")
    .reset_index()
)
df_demand = df_flows[df_flows.techs.str.contains("demand")]
df_flows_other = df_flows[~df_flows.techs.str.contains("demand")]

print(df_flows.head())

node_order = df_flows_other.nodes.unique()

fig = px.bar(
    df_flows_other,
    x="timesteps",
    y="Flow in/out (kWh)",
    facet_row="nodes",
    facet_col="carriers",
    color="techs",
    category_orders={"nodes": node_order, "carriers": carriers},
    height=1000,
    color_discrete_map=colors,
)

showlegend = True
# we reverse the node order (`[::-1]`) because the rows are numbered from bottom to top.
for row, node in enumerate(node_order[::-1]):
    for col, carrier in enumerate(carriers):
        demand_ = df_demand.loc[
            (df_demand.nodes == node) & (df_demand.techs == f"demand_{carrier}"),
            "Flow in/out (kWh)",
        ]
        if not demand_.empty:
            fig.add_scatter(
                x=model.results.timesteps.values,
                y=-1 * demand_,
                row=row + 1,
                col=col + 1,
                marker_color="black",
                name="Demand",
                legendgroup="demand",
                showlegend=showlegend,
            )
            showlegend = False
fig.update_yaxes(matches=None)
fig.show()

  nodes        techs     carriers  timesteps  Flow in/out (kWh)
0    D1    TH1_to_D1         heat 2050-01-01         500.000000
1    D1  demand_heat         heat 2050-01-01        -500.000000
2    D2    TH1_to_D2         heat 2050-01-01         500.000000
3    D2  demand_heat         heat 2050-01-01        -500.000000
4   SE1   SE1_to_SH1  electricity 2050-01-01        -302.451666


In [181]:
df_capacity = (
    model.results.flow_cap.where(
        ~model.inputs.base_tech.str.contains("demand|transmission")
    )
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow capacity (kW)")
    .reset_index()
)

print(df_capacity.head())

fig = px.bar(
    df_capacity,
    x="nodes",
    y="Flow capacity (kW)",
    color="techs",
    facet_col="carriers",
    color_discrete_map=colors,
)
fig.show()

  nodes               techs     carriers  Flow capacity (kW)
0   SE1  supply_electricity  electricity              2000.0
1   SH1   supply_geothermal         heat              2000.0


In [182]:
df_coords = model.inputs[["latitude", "longitude"]].to_dataframe().reset_index()
df_capacity = (
    model.results.flow_cap.where(model.inputs.base_tech == "transmission")
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow capacity (kW)")
    .reset_index()
)
df_capacity_coords = pd.merge(df_coords, df_capacity, left_on="nodes", right_on="nodes").sort_values(by=['techs'])
fig1 = px.line_map(
    df_capacity_coords,
    lat="latitude",
    lon="longitude",
    color="carriers",
    hover_name="nodes",
    hover_data="Flow capacity (kW)",
    zoom=3,
    height=2000,
)
fig2 = px.scatter_map(
    df_capacity_coords,
    lat="latitude",
    lon="longitude",
    color="carriers",
    hover_name="nodes",
    hover_data="Flow capacity (kW)",
    zoom=3,
    height=2000,
)
fig=go.Figure(data = fig1.data + fig2.data)
fig.update_layout(
    map_style="open-street-map",
    map_zoom=16,
    map_center_lat=df_coords.latitude.mean(),
    map_center_lon=df_coords.longitude.mean(),
    margin={"r": 0, "t": 0, "l": 0, "b": 0},
    hoverdistance=50,
)